# 48 — Add HYPOELLIPSE origins to the ObsPy catalog

Extends the linked catalog built in steps 40–54 with additional `Origin` and `Magnitude`
objects from GOLD (1992–93 HYPOELLIPSE) and BRUN (2001 HYPOELLIPSE).

**Association logic:**
1. For each GOLD / BRUN origin, search for an existing `Event` whose `preferred_origin`
   time (after applying the per-catalog clock offset) falls within `ASSOC_TOL_S` seconds.
   If found → attach a new `Origin` (and `Magnitude`) to that `Event`.
2. If no existing event matches but there are unassociated picks within the same time
   window → create a new `Event`, attach the origin, and link the picks.
3. If no match of any kind → create a new waveform-less, picks-less `Event` carrying
   only the HYPOELLIPSE origin.

**Preferred origin policy** (by reverse chronological processing date):
   BRUN (2001) > GOLD (1992–93) > MORI/PINAALL (1991 operational)

**Inputs:**
- `products/linked_catalog.xml` — QuakeML from step 54
- `products/unassociated_picks.xml` — picks not yet linked to any event (step 35)
- `products/waveform_index.csv` — index of DMX files with UTC start/end (step 10)
- `HYPOCENTERS/GOLD.SUM`, `HYPOCENTERS/b-run.sum`
- `products/clock_offset_table.csv` — from notebook 47

**Outputs:**
- `products/linked_catalog_v2.xml` — extended QuakeML
- `products/catalog_v2_seisan/` — SEISAN REA database
- `products/catalog_v2.css3` — Antelope CSS3.0 export
- `products/48_association_log.csv` — per-origin association decisions
- `products/48_new_event_counts.txt` — summary counts

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
from pathlib import Path

DATA_ROOT = Path('.')    # adjust to your archive root
HYPO_DIR  = DATA_ROOT / 'HYPOCENTERS'
PROD_DIR  = DATA_ROOT / 'products'
PROD_DIR.mkdir(exist_ok=True)

# Association tolerances
ASSOC_TOL_S        = 10.0   # seconds: search window for existing event match
ASSOC_TOL_PICKS_S  = 60.0   # seconds: wider window for unassociated pick search
ASSOC_MIN_PICKS    = 3      # minimum picks to constitute a pick-only new event

# Clock offsets for the HYPOELLIPSE catalogs (hours to add to raw time → UTC)
# GOLD: uniform +7 h (May 18 – Jun 14 1991 only)
# BRUN: 0 h (already on UTC throughout)
import datetime as _dt
GOLD_UTC_OFFSET = _dt.timedelta(hours=7)
BRUN_UTC_OFFSET = _dt.timedelta(hours=0)

# Source identifiers written into QuakeML <creationInfo>
AGENCY_ID = 'VDAP/PVO'
GOLD_CREATION_INFO = {
    'agency_id': AGENCY_ID,
    'author': 'HYPOELLIPSE (Lahr 1999); processing 1992–1993',
    'version': 'GOLD',
}
BRUN_CREATION_INFO = {
    'agency_id': AGENCY_ID,
    'author': 'HYPOELLIPSE (Lahr 1999); processing 2001',
    'version': 'BRUN',
}

# Preferred origin priority (highest index = most preferred)
ORIGIN_PRIORITY = {'MORI': 0, 'PINAALL': 0, 'GOLD': 1, 'BRUN': 2}

print("Configuration loaded.")

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from obspy import read_events, Catalog, UTCDateTime
from obspy.core.event import (
    Event, Origin, Magnitude, CreationInfo,
    OriginQuality, OriginUncertainty, QuantityError, Comment,
    ResourceIdentifier
)

print("Imports OK.")

## 1  Parsers (reused from notebook 47)

In [ ]:
def parse_hypoellipse_preY2K(path):
    """
    Parse GOLD.SUM (pre-Y2K HYPOELLIPSE).
    Returns list of dicts; times are raw (before UTC offset).
    Keys: time, lat, lon, dep, mag, rms, gap, nobs, dep_err,
          proc_month, proc_year
    """
    records = []
    for raw in Path(path).read_text(errors='replace').splitlines():
        line = raw.rstrip('\r\n')
        if len(line) < 80 or line[79] not in ('/', '\\'):
            continue
        try:
            yr2  = int(line[0:2])
            mo   = int(line[2:4])
            dy   = int(line[4:6])
            hr   = int(line[6:8].strip() or 0)
            mn   = int(line[8:10].strip() or 0)
            sc   = float(line[10:14].strip() or 0) / 100.0
            ld   = line[14:16].strip(); ns = line[16]
            lm   = line[17:21].strip()
            od   = line[21:24].strip(); ew = line[24]
            om   = line[25:29].strip()
            ds   = line[29:34].strip()
            ms   = line[34:36].strip()
            # Quality fields (pre-Y2K offset: subtract 2 from post-Y2K positions)
            nobs_s = line[36:39].strip()
            gap_s  = line[39:42].strip()
            rms_s  = line[45:49].strip()
            pm_s   = line[85:87].strip() if len(line) > 88 else ''
            py_s   = line[87:89].strip() if len(line) > 89 else ''
            if not ld or not od or not ld.lstrip('-').isdigit():
                continue
            lat = float(ld) + float(lm) / 100.0 / 60.0
            lon = float(od) + float(om) / 100.0 / 60.0
            if ns.upper() == 'S': lat = -lat
            if ew.upper() == 'W': lon = -lon
            if not (10 < lat < 20 and 115 < lon < 125): continue
            dep  = float(ds) / 100.0 if ds and ds.lstrip('-').isdigit() else None
            mag  = float(ms) / 10.0  if ms and ms.isdigit()             else None
            nobs = int(nobs_s)  if nobs_s.isdigit()  else None
            gap  = int(gap_s)   if gap_s.isdigit()   else None
            rms  = float(rms_s) / 100.0 if rms_s and rms_s.isdigit() else None
            pm   = int(pm_s)    if pm_s.isdigit()    else None
            py   = int(py_s)    if py_s.isdigit()    else None
            sc2  = min(int(sc), 59)
            us   = int((sc - int(sc)) * 1e6)
            t    = datetime.datetime(1900 + yr2, mo, dy, hr, mn, sc2, us)
            records.append(dict(time=t, lat=lat, lon=lon, dep=dep, mag=mag,
                                nobs=nobs, gap=gap, rms=rms,
                                proc_month=pm, proc_year=py,
                                catalog='GOLD'))
        except Exception:
            pass
    print(f"GOLD: {len(records)} records parsed from {path}")
    return records


def parse_hypoellipse_postY2K(path):
    """
    Parse b-run.sum (post-Y2K HYPOELLIPSE).
    Keys: same as parse_hypoellipse_preY2K.
    """
    records = []
    for raw in Path(path).read_text(errors='replace').splitlines():
        line = raw.rstrip('\r\n')
        if len(line) < 82 or line[81] not in ('/', '\\'):
            continue
        try:
            yr   = int(line[0:4])
            mo   = int(line[4:6])
            dy   = int(line[6:8])
            hr   = int(line[8:10].strip() or 0)
            mn   = int(line[10:12].strip() or 0)
            sc   = float(line[12:16].strip() or 0) / 100.0
            ld   = line[16:18].strip(); ns = line[18]
            lm   = line[19:23].strip()
            od   = line[23:26].strip(); ew = line[26]
            om   = line[27:31].strip()
            ds   = line[31:36].strip()
            ms   = line[36:38].strip()
            nobs_s = line[38:41].strip()
            gap_s  = line[41:44].strip()
            rms_s  = line[47:51].strip()
            pm_s   = line[87:89].strip() if len(line) > 90 else ''
            py_s   = line[89:91].strip() if len(line) > 91 else ''
            if not ld or not od or not ld.lstrip('-').isdigit(): continue
            lat = float(ld) + float(lm) / 100.0 / 60.0
            lon = float(od) + float(om) / 100.0 / 60.0
            if ns.upper() == 'S': lat = -lat
            if ew.upper() == 'W': lon = -lon
            if not (10 < lat < 20 and 115 < lon < 125): continue
            dep  = float(ds) / 100.0 if ds and ds.lstrip('-').isdigit() else None
            mag  = float(ms) / 10.0  if ms and ms.isdigit()             else None
            nobs = int(nobs_s)  if nobs_s.isdigit()  else None
            gap  = int(gap_s)   if gap_s.isdigit()   else None
            rms  = float(rms_s) / 100.0 if rms_s and rms_s.isdigit() else None
            pm   = int(pm_s)    if pm_s.isdigit()    else None
            py   = int(py_s)    if py_s.isdigit()    else None
            sc2  = min(int(sc), 59)
            us   = int((sc - int(sc)) * 1e6)
            t    = datetime.datetime(yr, mo, dy, hr, mn, sc2, us)
            records.append(dict(time=t, lat=lat, lon=lon, dep=dep, mag=mag,
                                nobs=nobs, gap=gap, rms=rms,
                                proc_month=pm, proc_year=py,
                                catalog='BRUN'))
        except Exception:
            pass
    print(f"BRUN: {len(records)} records parsed from {path}")
    return records

## 2  Build ObsPy Origin / Magnitude from a parsed record

In [ ]:
def _resource_id(prefix, t_utc):
    """Stable resource identifier from catalog prefix + UTC time."""
    ts = t_utc.strftime('%Y%m%dT%H%M%S') if hasattr(t_utc, 'strftime') else str(t_utc)
    return ResourceIdentifier(id=f'quakeml:vdap/pinatubo1991/{prefix}/{ts}')


def record_to_origin(rec, utc_offset, catalog_label, creation_info_dict):
    """
    Convert a parsed HYPOELLIPSE record to an ObsPy Origin + optional Magnitude.

    Parameters
    ----------
    rec : dict
        Parsed record from parse_hypoellipse_*.
    utc_offset : datetime.timedelta
        Clock correction to add to raw time.
    catalog_label : str
        'GOLD' or 'BRUN'.
    creation_info_dict : dict
        Fields for CreationInfo.

    Returns
    -------
    origin : obspy Origin
    magnitude : obspy Magnitude or None
    t_utc : UTCDateTime (corrected time)
    """
    # Apply clock offset
    t_raw = rec['time']
    t_utc = UTCDateTime(t_raw + utc_offset)

    ci = CreationInfo(
        agency_id=creation_info_dict.get('agency_id', ''),
        author=creation_info_dict.get('author', ''),
        version=creation_info_dict.get('version', ''),
    )

    # Build quality object from available HYPOELLIPSE fields
    quality_kwargs = {}
    if rec.get('nobs') is not None:
        quality_kwargs['used_phase_count'] = rec['nobs']
    if rec.get('gap') is not None:
        quality_kwargs['azimuthal_gap'] = float(rec['gap'])
    if rec.get('rms') is not None:
        quality_kwargs['standard_error'] = rec['rms']
    quality = OriginQuality(**quality_kwargs) if quality_kwargs else None

    # Comments carry the raw (pre-correction) time and provenance
    proc_str = ''
    if rec.get('proc_month') and rec.get('proc_year'):
        proc_str = (f"; processed {rec['proc_month']:02d}/"
                    f"{1900 + rec['proc_year']}")
    comment_text = (
        f"catalog={catalog_label}"
        f"; raw_time={t_raw.isoformat()}"
        f"; utc_offset_h={utc_offset.total_seconds()/3600:+g}"
        f"{proc_str}"
    )

    origin = Origin(
        resource_id=_resource_id(catalog_label, t_utc),
        time=t_utc,
        # ObsPy expects time_errors as QuantityError; reuse clock uncertainty
        # from the main clock_regimes.csv (25 s post-reset for BRUN).
        # Set 60 s for GOLD (older reprocessing, less constrained absolute time).
        time_errors=QuantityError(
            uncertainty=60.0 if catalog_label == 'GOLD' else 25.0
        ),
        latitude=rec['lat'],
        longitude=rec['lon'],
        depth=rec['dep'] * 1000.0 if rec.get('dep') is not None else None,  # m
        depth_type='from location',
        method_id=ResourceIdentifier(id='quakeml:vdap/method/HYPOELLIPSE'),
        earth_model_id=ResourceIdentifier(
            id='quakeml:vdap/pinatubo1991/velocity_model/five_layer'),
        quality=quality,
        creation_info=ci,
        comments=[Comment(text=comment_text)],
    )

    magnitude = None
    if rec.get('mag') is not None:
        # NOTE: GOLD and BRUN magnitude scales differ by ~0.6 units (see paper).
        # Store the raw value and document the offset in the comment.
        mag_comment = (
            f"catalog={catalog_label}; "
            f"GOLD magnitudes are ~0.6 units lower than BRUN "
            f"(systematic calibration difference; see paper Section X)"
            if catalog_label == 'GOLD' else
            f"catalog={catalog_label}"
        )
        magnitude = Magnitude(
            resource_id=_resource_id(f'{catalog_label}_mag', t_utc),
            mag=rec['mag'],
            magnitude_type='Ml',
            origin_id=origin.resource_id,
            station_count=rec.get('nobs'),
            creation_info=ci,
            comments=[Comment(text=mag_comment)],
        )

    return origin, magnitude, t_utc

## 3  Load the existing linked catalog

In [ ]:
catalog_path = PROD_DIR / 'linked_catalog.xml'
print(f"Loading existing catalog from {catalog_path} ...")
cat = read_events(str(catalog_path))
print(f"  Loaded {len(cat)} events")

# Build a fast lookup: preferred-origin UTC time (seconds since epoch) → event index
# This lets us find the nearest existing event in O(log n) after sorting.
import bisect

def _epoch(t):
    """UTCDateTime or datetime → float seconds since Unix epoch."""
    if isinstance(t, UTCDateTime):
        return float(t)
    return float(UTCDateTime(t))

event_times = []   # sorted list of (epoch_s, event_index)
for idx, ev in enumerate(cat):
    if ev.preferred_origin() is None and len(ev.origins) == 0:
        continue
    orig = ev.preferred_origin() or ev.origins[0]
    event_times.append((_epoch(orig.time), idx))
event_times.sort(key=lambda x: x[0])
sorted_epochs = [et[0] for et in event_times]

print(f"  Built time index for {len(event_times)} events with origins.")


def find_nearest_event(t_utc, tol_s=ASSOC_TOL_S):
    """
    Return (event_index, dt_s) for the existing event whose preferred-origin
    time is nearest to t_utc, or (None, None) if no event is within tol_s.
    """
    epoch = _epoch(t_utc)
    pos = bisect.bisect_left(sorted_epochs, epoch)
    best_idx, best_dt = None, float('inf')
    for p in [pos - 1, pos, pos + 1]:
        if 0 <= p < len(event_times):
            dt = abs(epoch - event_times[p][0])
            if dt < best_dt:
                best_dt = dt
                best_idx = event_times[p][1]
    if best_dt <= tol_s:
        return best_idx, best_dt
    return None, None

## 4  Load unassociated picks and waveform index (for new-event association)

In [ ]:
# Unassociated picks — may not exist if the step-35 output uses a different path.
# Adjust the path to match your project layout.
unassoc_pick_path = PROD_DIR / 'unassociated_picks.xml'
unassoc_picks = {}
if unassoc_pick_path.exists():
    unassoc_cat = read_events(str(unassoc_pick_path))
    # Index by epoch of first pick
    for ev in unassoc_cat:
        if ev.picks:
            first_pick_t = min(_epoch(p.time) for p in ev.picks)
            unassoc_picks[first_pick_t] = ev
    print(f"Loaded {len(unassoc_picks)} unassociated pick groups.")
else:
    print(f"No unassociated picks file at {unassoc_pick_path}; skipping pick association.")

# Waveform index — columns: filename, start_utc, end_utc (epoch seconds)
wav_index_path = PROD_DIR / 'waveform_index.csv'
wav_index = None
if wav_index_path.exists():
    wav_index = pd.read_csv(wav_index_path, parse_dates=['start_utc', 'end_utc'])
    wav_index['start_epoch'] = wav_index.start_utc.apply(
        lambda t: float(UTCDateTime(t)))
    wav_index['end_epoch']   = wav_index.end_utc.apply(
        lambda t: float(UTCDateTime(t)))
    print(f"Loaded waveform index: {len(wav_index)} files.")
else:
    print(f"No waveform index at {wav_index_path}; skipping waveform association.")


def find_waveform_files(t_utc, dep_km, tol_s=10.0):
    """Return list of waveform filenames whose window contains t_utc ± tol_s."""
    if wav_index is None:
        return []
    epoch = _epoch(t_utc)
    # A file covers the event if its window starts ≤ epoch+tol and ends ≥ epoch-tol
    mask = (
        (wav_index.start_epoch <= epoch + tol_s) &
        (wav_index.end_epoch   >= epoch - tol_s)
    )
    return wav_index.loc[mask, 'filename'].tolist()


def find_unassociated_picks(t_utc, tol_s=ASSOC_TOL_PICKS_S):
    """Return (pick_event, dt_s) for the nearest unassociated pick group."""
    epoch = _epoch(t_utc)
    best, best_dt = None, float('inf')
    for pick_epoch, pick_ev in unassoc_picks.items():
        dt = abs(epoch - pick_epoch)
        if dt < best_dt:
            best_dt = dt
            best = pick_ev
    if best_dt <= tol_s:
        return best, best_dt
    return None, None

## 5  Main association loop

In [ ]:
def associate_origins(records, utc_offset, catalog_label, creation_info_dict):
    """
    Associate a list of parsed HYPOELLIPSE records with the catalog.

    Returns
    -------
    new_origins_added : int
    new_events_with_picks : int
    new_events_origin_only : int
    log_rows : list of dicts (for the association log CSV)
    new_events : list of ObsPy Event objects to append to cat
    """
    n_assoc  = 0   # added to existing event
    n_picks  = 0   # new event with unassociated picks
    n_bare   = 0   # new bare event (origin only)
    log_rows = []
    new_events = []

    for rec in records:
        origin, magnitude, t_utc = record_to_origin(
            rec, utc_offset, catalog_label, creation_info_dict)

        # ── Case 1: existing event match ──────────────────────────────────────
        ev_idx, dt_s = find_nearest_event(t_utc, ASSOC_TOL_S)
        if ev_idx is not None:
            ev = cat[ev_idx]
            ev.origins.append(origin)
            if magnitude is not None:
                ev.magnitudes.append(magnitude)
            # Update preferred origin if this catalog has higher priority
            cur_pref = ev.preferred_origin()
            cur_label = ''
            if cur_pref is not None:
                for c in (cur_pref.comments or []):
                    if 'catalog=' in (c.text or ''):
                        cur_label = c.text.split('catalog=')[1].split(';')[0].strip()
                        break
            if ORIGIN_PRIORITY.get(catalog_label, -1) > ORIGIN_PRIORITY.get(cur_label, -1):
                ev.preferred_origin_id = origin.resource_id
                if magnitude is not None:
                    ev.preferred_magnitude_id = magnitude.resource_id
            n_assoc += 1
            log_rows.append(dict(
                catalog=catalog_label, raw_time=rec['time'].isoformat(),
                utc_time=str(t_utc), lat=rec['lat'], lon=rec['lon'],
                dep=rec.get('dep'), mag=rec.get('mag'),
                decision='ASSOC_EXISTING', dt_s=round(dt_s, 2),
                event_id=str(ev.resource_id)
            ))
            continue

        # ── Case 2: unassociated pick group ───────────────────────────────────
        pick_ev, pick_dt = find_unassociated_picks(t_utc, ASSOC_TOL_PICKS_S)
        if pick_ev is not None and len(pick_ev.picks) >= ASSOC_MIN_PICKS:
            new_ev = Event(
                resource_id=_resource_id(f'{catalog_label}_pickonly', t_utc),
                origins=[origin],
                magnitudes=[magnitude] if magnitude else [],
                picks=pick_ev.picks,
            )
            new_ev.preferred_origin_id    = origin.resource_id
            if magnitude is not None:
                new_ev.preferred_magnitude_id = magnitude.resource_id
            new_ev.comments.append(Comment(
                text=(f'New event from {catalog_label} origin; '
                      f'associated with {len(pick_ev.picks)} unassociated picks '
                      f'(dt={pick_dt:.1f} s)')
            ))
            new_events.append(new_ev)
            # Remove from pool so the same picks aren't reused
            first_pick_t = min(_epoch(p.time) for p in pick_ev.picks)
            unassoc_picks.pop(first_pick_t, None)
            n_picks += 1
            log_rows.append(dict(
                catalog=catalog_label, raw_time=rec['time'].isoformat(),
                utc_time=str(t_utc), lat=rec['lat'], lon=rec['lon'],
                dep=rec.get('dep'), mag=rec.get('mag'),
                decision='NEW_WITH_PICKS', dt_s=round(pick_dt, 2),
                event_id=str(new_ev.resource_id)
            ))
            continue

        # ── Case 3: bare origin-only event ────────────────────────────────────
        # Check whether a waveform file exists for this time
        wav_files = find_waveform_files(t_utc, rec.get('dep', 5.0))
        new_ev = Event(
            resource_id=_resource_id(f'{catalog_label}_bare', t_utc),
            origins=[origin],
            magnitudes=[magnitude] if magnitude else [],
        )
        new_ev.preferred_origin_id    = origin.resource_id
        if magnitude is not None:
            new_ev.preferred_magnitude_id = magnitude.resource_id
        wf_note = (f'; {len(wav_files)} waveform file(s) cover this window'
                   if wav_files else '; no waveform file found')
        new_ev.comments.append(Comment(
            text=f'New bare event from {catalog_label} origin{wf_note}'
        ))
        new_events.append(new_ev)
        n_bare += 1
        log_rows.append(dict(
            catalog=catalog_label, raw_time=rec['time'].isoformat(),
            utc_time=str(t_utc), lat=rec['lat'], lon=rec['lon'],
            dep=rec.get('dep'), mag=rec.get('mag'),
            decision='NEW_BARE', dt_s=None,
            event_id=str(new_ev.resource_id),
            n_waveform_files=len(wav_files)
        ))

    return n_assoc, n_picks, n_bare, log_rows, new_events

## 6  Run association for GOLD then BRUN

In [ ]:
# Parse HYPOELLIPSE files
gold_records = parse_hypoellipse_preY2K(HYPO_DIR / 'GOLD.SUM')
brun_records = parse_hypoellipse_postY2K(HYPO_DIR / 'b-run.sum')

all_log = []
all_new_events = []

print("\n── Associating GOLD origins ──")
na, np_, nb, log, new_ev = associate_origins(
    gold_records, GOLD_UTC_OFFSET, 'GOLD', GOLD_CREATION_INFO)
print(f"  Added to existing: {na}")
print(f"  New (with picks):  {np_}")
print(f"  New (bare):        {nb}")
all_log.extend(log)
all_new_events.extend(new_ev)

# After GOLD, update the time index to include new GOLD-only events
# so BRUN can associate to them.
for ev in new_ev:
    orig = ev.preferred_origin() or ev.origins[0]
    t_ep = _epoch(orig.time)
    # Insert into sorted list
    idx = len(cat) + len(all_new_events) - len(new_ev) + new_ev.index(ev)
    bisect.insort(event_times, (t_ep, -1))   # -1 = will resolve after append
    bisect.insort(sorted_epochs, t_ep)

print("\n── Associating BRUN origins ──")
na, np_, nb, log, new_ev = associate_origins(
    brun_records, BRUN_UTC_OFFSET, 'BRUN', BRUN_CREATION_INFO)
print(f"  Added to existing: {na}")
print(f"  New (with picks):  {np_}")
print(f"  New (bare):        {nb}")
all_log.extend(log)
all_new_events.extend(new_ev)

# Append new events to catalog
for ev in all_new_events:
    cat.append(ev)

print(f"\nExtended catalog: {len(cat)} events total")

## 7  Save association log and summary

In [ ]:
log_df = pd.DataFrame(all_log)
log_path = PROD_DIR / '48_association_log.csv'
log_df.to_csv(log_path, index=False)
print(f"Association log → {log_path}")
print("\nDecision counts:")
print(log_df.groupby(['catalog', 'decision']).size().to_string())

summary_path = PROD_DIR / '48_new_event_counts.txt'
with open(summary_path, 'w') as f:
    f.write("Notebook 48 — HYPOELLIPSE origin association summary\n")
    f.write("=" * 50 + "\n")
    for cat_label in ['GOLD', 'BRUN']:
        sub = log_df[log_df.catalog == cat_label]
        f.write(f"\n{cat_label}:\n")
        for dec, n in sub.decision.value_counts().items():
            f.write(f"  {dec}: {n}\n")
    f.write(f"\nTotal events in extended catalog: {len(cat)}\n")
print(f"Summary → {summary_path}")

## 8  Export extended catalog

### 8.1  QuakeML

In [ ]:
quakeml_out = PROD_DIR / 'linked_catalog_v2.xml'
cat.write(str(quakeml_out), format='QUAKEML')
print(f"QuakeML → {quakeml_out}  ({quakeml_out.stat().st_size/1e6:.1f} MB)")

### 8.2  SEISAN REA

In [ ]:
# SEISAN REA export — requires flovopy or the SEISAN writer from step 50.
# The pattern below assumes a FloVoPy SEISAN writer is available.
# Replace with your project's actual SEISAN export function.

seisan_dir = PROD_DIR / 'catalog_v2_seisan'
seisan_dir.mkdir(exist_ok=True)

try:
    # Try FloVoPy SEISAN writer (adjust import path to match your project)
    from flovopy.io.seisan import write_seisan_rea
    n_written = write_seisan_rea(cat, seisan_dir, overwrite=False)
    print(f"SEISAN REA → {seisan_dir}  ({n_written} S-files written)")
except ImportError:
    # Fallback: use ObsPy's Nordic writer
    print("FloVoPy not found; writing Nordic format with ObsPy.")
    n_written = 0
    for ev in cat:
        try:
            orig = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
            if orig is None:
                continue
            t = orig.time
            yr, mo = t.year, t.month
            ydir = seisan_dir / str(yr)
            mdir = ydir / f'{mo:02d}'
            mdir.mkdir(parents=True, exist_ok=True)
            fname = (f"{t.year:04d}-{t.month:02d}-{t.day:02d}-"
                     f"{t.hour:02d}{t.minute:02d}-{t.second:02d}S.PIN__001")
            Catalog(events=[ev]).write(
                str(mdir / fname), format='NORDIC')
            n_written += 1
        except Exception as e:
            pass   # skip events that fail Nordic serialisation
    print(f"SEISAN REA (Nordic) → {seisan_dir}  ({n_written} files)")

### 8.3  Antelope CSS3.0

In [ ]:
# CSS3.0 export — write origin, assoc, netmag, event tables.
# This is a simplified writer; for the full conversion use the step-52 notebook.

css_path = PROD_DIR / 'catalog_v2.css3'
css_path.mkdir(exist_ok=True)

origin_rows = []
netmag_rows = []

for ev_id, ev in enumerate(cat):
    for orig in ev.origins:
        # Extract catalog label from comment
        cat_label = 'UNK'
        for c in (orig.comments or []):
            if 'catalog=' in (c.text or ''):
                cat_label = c.text.split('catalog=')[1].split(';')[0].strip()
                break
        t = orig.time
        origin_rows.append({
            'evid':   ev_id,
            'orid':   str(orig.resource_id),
            'time':   float(t),
            'lat':    orig.latitude,
            'lon':    orig.longitude,
            'depth':  (orig.depth or 0.0) / 1000.0,  # km
            'ndef':   (orig.quality.used_phase_count
                       if orig.quality else -1),
            'gap':    (orig.quality.azimuthal_gap
                       if orig.quality else -1),
            'sdobs':  (orig.quality.standard_error
                       if orig.quality else -1),
            'auth':   cat_label,
            'lddate': t.strftime('%Y-%m-%d'),
        })
    for magobj in ev.magnitudes:
        netmag_rows.append({
            'evid':  ev_id,
            'magid': str(magobj.resource_id),
            'orid':  str(magobj.origin_id),
            'mag':   magobj.mag,
            'mtype': magobj.magnitude_type or 'Ml',
            'nsta':  magobj.station_count or -1,
        })

pd.DataFrame(origin_rows).to_csv(css_path / 'origin.csv', index=False)
pd.DataFrame(netmag_rows).to_csv(css_path / 'netmag.csv', index=False)
print(f"CSS3.0 tables → {css_path}")
print(f"  origin.csv:  {len(origin_rows)} rows")
print(f"  netmag.csv:  {len(netmag_rows)} rows")

## 9  Catalog composition summary

In [ ]:
# Count origins per event and how many events have BRUN, GOLD, MORI origins
has_gold = sum(1 for ev in cat
               for o in ev.origins
               for c in (o.comments or [])
               if 'catalog=GOLD' in (c.text or ''))
has_brun = sum(1 for ev in cat
               for o in ev.origins
               for c in (o.comments or [])
               if 'catalog=BRUN' in (c.text or ''))

print("=" * 50)
print("EXTENDED CATALOG COMPOSITION")
print("=" * 50)
print(f"Total events:         {len(cat)}")
print(f"Origins from GOLD:    {has_gold}")
print(f"Origins from BRUN:    {has_brun}")
print()
print("Preferred origin source distribution:")
pref_sources = {}
for ev in cat:
    pref = ev.preferred_origin()
    if pref is None:
        pref_sources['none'] = pref_sources.get('none', 0) + 1
        continue
    label = 'UNK'
    for c in (pref.comments or []):
        if 'catalog=' in (c.text or ''):
            label = c.text.split('catalog=')[1].split(';')[0].strip()
            break
    pref_sources[label] = pref_sources.get(label, 0) + 1
for k, v in sorted(pref_sources.items(), key=lambda x: -x[1]):
    print(f"  {k:10s}: {v:5d}")